# Stage 11 — Option buying ranking

Full-universe backtest (notebook 15, WITH the 4H hard filter applied) showed 70.6% of qualified instruments with a positive avg R, mean 0.198, median 0.170. This ranks instruments the same way the other four strategies are ranked (`strategy_scoring.py`), so the shortlist is the instruments where this specific signal+filter+lifecycle actually worked, not the whole universe.

**`min_trades=20` here, not the 30 used for debit spread** - the 4H filter roughly halves signal frequency, so per-instrument closed_trades over the same window now tops out around 27 with a mean of ~19; a 30-trade bar would leave nothing ranked. See `score_for_option_buying()`'s docstring for the full reasoning.

median_r_achieved is weighted highest (not avg_r_achieved) - same median-over-mean preference as the rest of this project's stats.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.strategy_scoring import score_for_option_buying, save_ranking

comparison = pd.read_excel("../exports/OPTION_BUYING_COMPARISON.xlsx", sheet_name="Comparison")
comparison.shape

(214, 8)

In [2]:
ranked, watchlist = score_for_option_buying(comparison)
ranked.head(20)

,symbol,option_buying_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
0,PFC,92.7,,26,69.2,0.72,0.62,6.5,69.2
1,RVNL,92.0,,42,73.8,0.79,0.68,7.0,71.4
2,GMRAIRPORT,91.9,,34,73.5,0.52,0.39,6.0,73.5
3,ADANIENSOL,91.2,,33,63.6,0.49,0.40,5.0,63.6
4,COFORGE,91.2,,42,64.3,0.68,0.42,6.0,64.3
5,KPITTECH,90.9,,37,75.7,0.58,0.62,7.0,70.3
6,GODREJPROP,89.4,,32,71.9,0.48,0.32,6.0,68.8
7,BDL,88.9,,38,68.4,0.56,0.30,6.0,60.5
8,HINDUNILVR,87.8,,37,64.9,0.86,0.77,9.0,56.8
9,SUZLON,87.7,,37,59.5,0.72,0.48,6.0,59.5


In [3]:
# How many top-20 candidates carry the outlier-reliance warning?
print(f"Instruments passing min_trades filter: {len(ranked)} of {len(comparison)}")
print(f"Flagged in top 20: {(ranked.head(20)['outlier_reliance_flag'] != '').sum()}")
ranked[ranked["outlier_reliance_flag"] != ""].head(10)

Instruments passing min_trades filter: 211 of 214
Flagged in top 20: 1


,symbol,option_buying_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
15,RECLTD,85.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.40R),31,67.7,0.63,0.23,6.0,64.5
25,PAYTM,81.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.33R),34,61.8,0.63,0.30,7.0,58.8
26,POWERINDIA,80.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.36R),45,64.4,0.61,0.25,7.0,57.8
29,SHRIRAMFIN,80.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.61R),38,68.4,0.81,0.20,7.0,63.2
37,OIL,75.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.39R),31,61.3,0.60,0.21,7.0,61.3
39,BANKINDIA,75.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.33R),32,62.5,0.55,0.22,7.5,62.5
40,INFY,73.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.45R),42,61.9,0.65,0.20,8.0,57.1
63,VOLTAS,64.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.31R),28,57.1,0.48,0.17,9.0,57.1
64,AMBER,64.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.34R),37,56.8,0.45,0.11,6.0,54.1
96,NMDC,55.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.37R),33,54.5,0.42,0.05,6.0,54.5


In [4]:
save_ranking(ranked, "RANKING_option_buying.xlsx")

Saved ranking to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/RANKING_option_buying.xlsx


'/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/RANKING_option_buying.xlsx'

In [5]:
# Instruments below min_trades - not blended into the percentile ranking,
# but not dropped either. Sorted by avg_r_achieved. Re-check at the next
# periodic refresh - trade count grows on its own as more calendar time passes.
print(f"Instruments on watchlist (below min_trades): {len(watchlist)}")
watchlist

Instruments on watchlist (below min_trades): 3


,symbol,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
0,SWIGGY,18,83.3,1.21,0.94,6.0,77.8
1,WAAREEENER,18,61.1,0.24,0.21,6.0,55.6
2,HYUNDAI,16,56.2,-0.11,0.04,8.0,56.2


In [6]:
save_ranking(watchlist, "WATCHLIST_option_buying.xlsx")

Saved ranking to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/WATCHLIST_option_buying.xlsx


'/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/WATCHLIST_option_buying.xlsx'